# Certified Numerical Analysis with Rocq and AI Assistance

Workshop target: 90 minutes for mathematicians at a Math & AI event.

We use one running example:

$$ f(x) = \operatorname{sech}(10x-2)^2 + \operatorname{sech}(100x-40)^4 + \operatorname{sech}(1000x-600)^6. $$

A naive black-box numerical integration can miss the very narrow last bump. The Mathematica call

```Mathematica
Clear[x, f];
f[x_] := Sech[10 x - 2]^2 + Sech[100 x - 40]^4 + Sech[1000 x - 600]^6;
NIntegrate[f[x], {x, 0, 1}]
```

return `0.209736`, while Rocq will certify the first four decimals as `0.2108`.

The first part uses no LLM. After the certificate, we switch to an AI-aided proof workflow: retrieve relevant Rocq facts, ask an external model for proof scripts, test them, and introduce intermediate lemmas when the proof gets stuck.

## Session Map

- 0-10 min: the numerical-analysis problem and why formal certification matters.
- 10-30 min: Rocq through a tiny Python API: imports, definitions, theorem, tactics.
- 30-40 min: the interval tactic certifies the four decimals.
- 40-55 min: the analytic antiderivative plan.
- 55-80 min: retrieval + LLM suggestions + proof-state feedback.
- 80-90 min: final theorem, discussion, and limitations.

## Colab Runtime Setup

The notebook is intended to run in Google Colab. The first setup cell installs this repository as a Python package from GitHub using its `pyproject.toml`, including the Colab extras needed for retrieval and the Rocq client.

For the workshop, `rocq-ml-server` should run on a remote machine from the container image `theostos/coq-coquelicot:8.20-3.4.4`. Colab only runs the Python client. Set `ROCQ_SERVER_HOST` below to the public host or tunnel endpoint for that machine; all Rocq interaction then goes through `http://ROCQ_SERVER_HOST:5000`.

Retrieval is local to the Colab runtime: the precomputed FAISS index is downloaded into `/content`, then all documentation queries are served inside the notebook process. The notebook requires a real cache; there is no remote semantic service and no lexical fallback.


In [ ]:
%pip install -q integral-tp[colab]@git+https://github.com/theostos/integral-tp.git

In [ ]:
import os
import sys

import requests

IN_COLAB = "google.colab" in sys.modules or bool(os.getenv("COLAB_RELEASE_TAG"))

from workshop_api import (
    DEFAULT_EMBEDDING_MODEL,
    LLMClient,
    RetrievalClient,
    RocqWorkshop,
    format_retrieval_hits,
    prepare_colab_retrieval_cache,
    proof_prompt,
)

### Connect To The Rocq Server

Replace the host below by the remote machine that runs `rocq-ml-server`. For local testing, keep `127.0.0.1` and start the server yourself.


In [ ]:
ROCQ_SERVER_HOST = os.getenv("ROCQ_SERVER_HOST", "127.0.0.1")  # e.g. "rocq-workshop.example.org"
ROCQ_SERVER_PORT = int(os.getenv("ROCQ_SERVER_PORT", "5000"))
ROCQ_HEALTH_URL = f"http://{ROCQ_SERVER_HOST}:{ROCQ_SERVER_PORT}/health"

try:
    response = requests.get(ROCQ_HEALTH_URL, timeout=5)
    print(f"{ROCQ_HEALTH_URL} -> HTTP {response.status_code}")
    print(response.text[:500])
except requests.RequestException as exc:
    print(f"Could not reach {ROCQ_HEALTH_URL}: {exc}")
    print("Set ROCQ_SERVER_HOST/ROCQ_SERVER_PORT to the remote workshop server before continuing.")

rocq = RocqWorkshop(host=ROCQ_SERVER_HOST, port=ROCQ_SERVER_PORT, timeout=600)
rocq.root_path

### Retrieval Cache Setup

Paste the direct download URL for the precomputed `retrieval_cache.zip` below. The cache contains FAISS embeddings for Stdlib and Coquelicot docstrings/statements. The default embedding model is `Qwen/Qwen3-Embedding-4B`, and the query-side model must match the model recorded in the cache manifest.


In [ ]:
DOCSTRING_CACHE_URL = os.getenv("DOCSTRING_CACHE_URL", "").strip()
DOCSTRING_CACHE_DIR = os.getenv("DOCSTRING_CACHE_DIR", "").strip()
# DOCSTRING_CACHE_URL = "https://your-direct-download-link/retrieval_cache.zip"

if DOCSTRING_CACHE_URL:
    cache_path = prepare_colab_retrieval_cache(DOCSTRING_CACHE_URL, cache_dir="/content/rocq-doc-cache")
elif DOCSTRING_CACHE_DIR:
    cache_path = DOCSTRING_CACHE_DIR
else:
    raise RuntimeError(
        "Set DOCSTRING_CACHE_URL to the hosted retrieval_cache.zip, "
        "or set DOCSTRING_CACHE_DIR to an already-unpacked local cache."
    )

retriever = RetrievalClient.from_env(cache_dir=cache_path)
print(f"Retrieval cache: {cache_path}")
print(f"Default embedding model: {DEFAULT_EMBEDDING_MODEL}")

Run two queries before any LLM call. The first is restricted to Coquelicot, the second to Stdlib. This is useful in practice: the mathematical integration facts and the algebra/tactic facts often live in different libraries.


In [ ]:
coquelicot_hits = retriever.search(
    "RInt fundamental theorem derivative continuous interval integral",
    library="Coquelicot",
    k=5,
)
print("Coquelicot hits:")
print(format_retrieval_hits(coquelicot_hits))

stdlib_hits = retriever.search(
    "field_simplify ring exponential positivity exp_plus denominator nonzero",
    library="Stdlib",
    k=5,
)
print("\nStdlib hits:")
print(format_retrieval_hits(stdlib_hits))

## Part 1: Build the Rocq Environment

The first Rocq commands are just library imports. Mathematically, this says which language and which facts/tactics are available.

In [ ]:
for command in [
    "From Stdlib Require Import Reals Lra Psatz.",
    "From Coquelicot Require Import Coquelicot.",
    "From Interval Require Import Tactic.",
]:
    print(rocq.add_element(command)["kind"], command)

Now define the function and the integral. Rocq's `R` is the exact real numbers, not floating-point numbers. `RInt f 0 1` is the Coquelicot integral.

In [ ]:
rocq.add_definition(r'''
Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).
''')

rocq.add_definition(r'''
Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.
''')

rocq.add_definition("Definition I : R := RInt f 0 1.")

## Part 2: Certified Four Decimals, Without an LLM

The theorem below means that the exact real number `I` lies in the interval `[0.2107, 0.2109]`. In particular, the first four decimal digits are `0.2108` up to the stated tolerance.

In [ ]:
rocq.add_lemma("Theorem I_first_4_decimal_digits : Rabs (I - 0.2108) <= 1e-4.")

In [ ]:
rocq.run_tac("I_first_4_decimal_digits", "unfold I, f, sech.")

In [ ]:
rocq.run_tac(
    "I_first_4_decimal_digits",
    "integral with (i_prec 25, i_degree 3, i_fuel 300).",
)

In [ ]:
rocq.complete_lemma("I_first_4_decimal_digits")

In [ ]:
print(rocq.lemma_source("I_first_4_decimal_digits"))

At this point we have used no LLM. The trusted result is a kernel-checked Rocq theorem. The heavy work is delegated to the `interval` tactic, which builds a certificate that Rocq checks.

## Part 3: Analytic Plan

For the AI-aided part, we prove a closed form. The substitution is classical:

$$ \tanh(u) = \frac{e^{2u}-1}{e^{2u}+1}, \qquad \operatorname{sech}(u)^2 = 1 - \tanh(u)^2. $$

Then

$$ \int \operatorname{sech}(u)^2 du = \tanh(u), $$
$$ \int \operatorname{sech}(u)^4 du = \tanh(u) - \tfrac13 \tanh(u)^3, $$
$$ \int \operatorname{sech}(u)^6 du = \tanh(u) - \tfrac23 \tanh(u)^3 + \tfrac15 \tanh(u)^5. $$

In [ ]:
for command in [
    r'''
Definition tanhE (u : R) : R :=
  (exp (2 * u) - 1) / (exp (2 * u) + 1).
''',
    "Definition A2 (u : R) : R := tanhE u.",
    r'''
Definition A4 (u : R) : R :=
  tanhE u - (/ 3) * (tanhE u)^3.
''',
    r'''
Definition A6 (u : R) : R :=
  tanhE u - (2 / 3) * (tanhE u)^3 + (/ 5) * (tanhE u)^5.
''',
    r'''
Definition F2 (x : R) : R :=
  A2 (10 * x - 2) / 10.
''',
    r'''
Definition F4 (x : R) : R :=
  A4 (100 * x - 40) / 100.
''',
    r'''
Definition F6 (x : R) : R :=
  A6 (1000 * x - 600) / 1000.
''',
    r'''
Definition F (x : R) : R :=
  F2 x + F4 x + F6 x.
''',
    "Definition I_closed_form : R := F 1 - F 0.",
]:
    rocq.add_element(command)

## Part 4: First Manual Lemma

Before using AI, prove one simple lemma by hand. This also introduces the shape of a proof state: a goal, hypotheses, and tactics that transform the goal.

In [ ]:
rocq.add_lemma(r'''
Lemma sech_denominator_nonzero (u : R) :
  exp (2 * u) + 1 <> 0.
''')

In [ ]:
rocq.run_tac("sech_denominator_nonzero", r'''
apply Rgt_not_eq.
apply Rplus_lt_0_compat; [apply exp_pos | lra].
''')

In [ ]:
rocq.complete_lemma("sech_denominator_nonzero")

## Part 5: Retrieval Before Asking an LLM

From here onward, LLM calls are allowed. The discipline is:

1. inspect the current proof state;
2. retrieve relevant Rocq facts/tactics from documentation/docstrings;
3. ask the LLM for a proof script using that context;
4. test the script in Rocq;
5. if it fails, use the error or goal shape to introduce a smaller lemma.

We reuse the `retriever` created in the setup section. With the full FAISS cache, this searches Stdlib and Coquelicot locally inside the Colab runtime.


In [ ]:
hits = retriever.search(
    "differentiate quotient exponential field_simplify denominator nonzero exp_plus",
    k=8,
)
print(format_retrieval_hits(hits))

Retrieval is intentionally local-only: `RetrievalClient` loads a FAISS cache from `DOCSTRING_CACHE_DIR` or downloads one from `DOCSTRING_CACHE_URL`. If neither is configured, retrieval fails immediately. Use `library="Stdlib"` / `library="Coquelicot"` and `kind="definition"` / `kind=["definition", "start_theorem_proof"]` in `retriever.search` when you want to restrict the metadata before returning hits.


In [ ]:
rocq.add_lemma(r'''
Lemma tanhE_derivative (u : R) :
  is_derive tanhE u (sech u ^ 2).
''')
rocq.goals("tanhE_derivative")

In [ ]:
llm = LLMClient.from_env()
prompt = proof_prompt(
    lemma_header=rocq.lemmas["tanhE_derivative"].header,
    goals=rocq.goals("tanhE_derivative")["goals"],
    retrieval_hits=hits,
    extra_context="Definitions sech and tanhE have just been unfolded in similar hand proofs.",
)

if llm.configured:
    print(llm.chat(
        system="You help write short Rocq proof scripts for real analysis. Return tactics only.",
        user=prompt,
    ))
else:
    print("No LLM key configured. Here is the prompt that would be sent:\n")
    print(prompt)

We now test a proof script. The important point is not whether the model guessed it exactly, but that Rocq gives immediate feedback and keeps us honest.

In [ ]:
rocq.run_tac("tanhE_derivative", r'''
unfold tanhE, sech.
auto_derive.
- apply sech_denominator_nonzero.
- field_simplify.
  + replace (2 * u) with (u + u) by ring.
    rewrite exp_plus.
    replace (exp u * exp u) with (exp u ^ 2) by ring.
    reflexivity.
  + apply sech_denominator_nonzero.
  + apply sech_denominator_nonzero.
''')

In [ ]:
rocq.complete_lemma("tanhE_derivative")

The proof used `auto_derive` to compute the derivative and `field_simplify` plus `exp_plus` to reconcile the algebraic form of `sech`.

In [ ]:
def prove(header, steps):
    info = rocq.add_lemma(header)
    name = info["lemma"]
    for step in steps:
        out = rocq.run_tac(name, step)
        if not out["ok"]:
            raise RuntimeError(out)
    done = rocq.complete_lemma(name)
    if not done["ok"]:
        raise RuntimeError(done)
    return done

prove(r'''
Lemma A2_derivative (u : R) :
  is_derive A2 u (sech u ^ 2).
''', [r'''
unfold A2.
apply tanhE_derivative.
'''])

## Part 6: Getting Stuck, Then Introducing a Sublemma

We now start the proof of the derivative of `A4`. After symbolic differentiation, a purely algebraic identity appears. This is the moment to pause and prove a reusable helper lemma.

In [ ]:
rocq.add_lemma(r'''
Lemma A4_derivative (u : R) :
  is_derive A4 u (sech u ^ 4).
''')

rocq.run_tac("A4_derivative", r'''
unfold A4.
auto_derive.
''')

In [ ]:
rocq.goals("A4_derivative")

The second goal is an algebraic simplification involving `sech u ^ 2` and `tanhE u ^ 2`. Ask retrieval for the relevant identity.

In [ ]:
identity_hits = retriever.search(
    "identity sech squared one minus tanh squared exponential",
    k=8,
)
print(format_retrieval_hits(identity_hits))

In [ ]:
prove(r'''
Lemma sech_tanhE_identity (u : R) :
  sech u ^ 2 = 1 - tanhE u ^ 2.
''', [r'''
unfold sech, tanhE.
field_simplify.
- replace (2 * u) with (u + u) by ring.
  rewrite exp_plus.
  replace (exp u * exp u) with (exp u ^ 2) by ring.
  ring.
- apply sech_denominator_nonzero.
- apply sech_denominator_nonzero.
'''])

Completing the helper refreshes the open `A4_derivative` proof by replaying its existing steps in the stronger environment. We can now finish it.

In [ ]:
rocq.run_tac("A4_derivative", r'''
- repeat split.
  + exists (sech u ^ 2); apply tanhE_derivative.
  + exists (sech u ^ 2); apply tanhE_derivative.
- rewrite (is_derive_unique (fun x : R => tanhE x) u (sech u ^ 2)).
  + replace (sech u ^ 4) with ((sech u ^ 2) ^ 2) by ring.
    repeat rewrite sech_tanhE_identity.
    field.
  + apply tanhE_derivative.
''')

In [ ]:
rocq.complete_lemma("A4_derivative")

## Part 7: Fast-Forward the Remaining Derivative Lemmas

The remaining lemmas repeat the same pattern: symbolic differentiation, a uniqueness rewrite for `Derive`, and algebra using `sech_tanhE_identity`.

In [ ]:
prove(r'''
Lemma A6_derivative (u : R) :
  is_derive A6 u (sech u ^ 6).
''', [r'''
unfold A6.
auto_derive.
- repeat split.
  + exists (sech u ^ 2); apply tanhE_derivative.
  + exists (sech u ^ 2); apply tanhE_derivative.
  + exists (sech u ^ 2); apply tanhE_derivative.
- rewrite (is_derive_unique (fun x : R => tanhE x) u (sech u ^ 2)).
  + replace (sech u ^ 6) with ((sech u ^ 2) ^ 3) by ring.
    repeat rewrite sech_tanhE_identity.
    field.
  + apply tanhE_derivative.
'''])

In [ ]:
prove(r'''
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
''', [r'''
unfold F2.
auto_derive.
- exists (sech (10 * x + - (2)) ^ 2).
  apply A2_derivative.
- replace (Derive (fun y : R => A2 y) (10 * x + - (2)))
    with (sech (10 * x + - (2)) ^ 2).
  + replace (10 * x + - (2)) with (10 * x - 2) by ring.
    field.
  + symmetry.
    apply is_derive_unique.
    apply A2_derivative.
'''])

In [ ]:
prove(r'''
Lemma F4_derivative (x : R) :
  is_derive F4 x ((sech (100 * x - 40)) ^ 4).
''', [r'''
unfold F4.
auto_derive.
- exists (sech (100 * x + - (40)) ^ 4).
  apply A4_derivative.
- replace (Derive (fun y : R => A4 y) (100 * x + - (40)))
    with (sech (100 * x + - (40)) ^ 4).
  + replace (100 * x + - (40)) with (100 * x - 40) by ring.
    field.
  + symmetry.
    apply is_derive_unique.
    apply A4_derivative.
'''])

In [ ]:
prove(r'''
Lemma F6_derivative (x : R) :
  is_derive F6 x ((sech (1000 * x - 600)) ^ 6).
''', [r'''
unfold F6.
auto_derive.
- exists (sech (1000 * x + - (600)) ^ 6).
  apply A6_derivative.
- replace (Derive (fun y : R => A6 y) (1000 * x + - (600)))
    with (sech (1000 * x + - (600)) ^ 6).
  + replace (1000 * x + - (600)) with (1000 * x - 600) by ring.
    field.
  + symmetry.
    apply is_derive_unique.
    apply A6_derivative.
'''])

## Part 8: Assemble the Antiderivative Proof

Now the derivative of `F` is exactly `f`. The rest is the fundamental theorem of calculus in Coquelicot.

In [ ]:
prove(r'''
Lemma F_derivative (x : R) :
  is_derive F x (f x).
''', [r'''
unfold F, f.
apply (is_derive_plus 
  (fun y : R => F2 y + F4 y) F6 x
  ((sech (10 * x - 2)) ^ 2 + (sech (100 * x - 40)) ^ 4)
  ((sech (1000 * x - 600)) ^ 6)).
- apply (is_derive_plus F2 F4 x
    ((sech (10 * x - 2)) ^ 2)
    ((sech (100 * x - 40)) ^ 4));
    [apply F2_derivative | apply F4_derivative].
- apply F6_derivative.
'''])

In [ ]:
prove(r'''
Lemma f_ex_derive (x : R) :
  ex_derive f x.
''', [r'''
unfold f, sech.
auto_derive.
repeat split; try exact Logic.I.
all: apply sech_denominator_nonzero.
'''])

prove(r'''
Lemma f_continuous (x : R) :
  continuous f x.
''', [r'''
apply (ex_derive_continuous f x).
apply f_ex_derive.
'''])

In [ ]:
prove(r'''
Theorem I_closed_form_correct : I = I_closed_form.
''', [r'''
unfold I, I_closed_form.
apply is_RInt_unique.
apply (is_RInt_derive F f 0 1).
- intros x _. apply F_derivative.
- intros x _. apply f_continuous.
'''])

## Final Source

The Python session can materialize the Rocq development it built interactively.

In [ ]:
final_source = rocq.source(include_open=False)
print(final_source[:4000])
print("\n... total characters:", len(final_source))

## Discussion Prompts

- Which parts were numerical, and which parts were symbolic?
- Where did Rocq need mathematical insight rather than computation?
- What did retrieval add before calling the LLM?
- Which failures were useful because they suggested a better intermediate lemma?
- What would you want the AI assistant to automate, and what should remain explicit?